# Vietnamese legal NER with NlpHUST

This notebook pulls the inference script from the VDT-Anonymization GitHub repository and runs it over a Kaggle-attached copy of `legal_test.jsonl`. The script uses sliding windows so long legal documents are not silently truncated.

The legal dataset and model weights are intentionally not stored in GitHub. Attach them to Kaggle as datasets and set their paths below.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
from collections import Counter

# ---- Edit these values for the Kaggle datasets attached to your notebook ----
REPO_URL = "https://github.com/no1ceboy/VDT-Anonymization.git"
REPO_DIR = Path("/kaggle/working/VDT-Anonymization")

# Example: /kaggle/input/vdt-legal/legal_test.jsonl
INPUT_FILE = Path("/kaggle/input/REPLACE_DATASET_NAME/legal_test.jsonl")

# Option A: Kaggle Internet enabled; Transformers downloads/caches this model.
MODEL_SOURCE = "huggingface"
MODEL_NAME = "NlpHUST/ner-vietnamese-electra-base"

# Option B: attach model files as a Kaggle Dataset and use a local path.
MODEL_PATH = Path("/kaggle/input/REPLACE_MODEL_DATASET/ner-vietnamese-electra-base")

OUTPUT_DIR = Path("/kaggle/working/ner_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_OUTPUT = OUTPUT_DIR / "nlphust_legal_sample.jsonl"
FULL_OUTPUT = OUTPUT_DIR / "nlphust_legal_per_loc.jsonl"

# Keep this small for the first run. Set to 0 only after inspecting results.
SAMPLE_LIMIT = 20
RUN_FULL_DATASET = False

print("Input exists:", INPUT_FILE.exists(), INPUT_FILE)
print("Repository target:", REPO_DIR)

In [ ]:
# Pull the current inference code. No package installation is needed here.
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

SCRIPT = REPO_DIR / "src" / "run_ner.py"
assert SCRIPT.exists(), f"Inference script not found: {SCRIPT}"
print(SCRIPT)

In [ ]:
def make_command(output_file, limit):
    command = [
        "python", str(SCRIPT),
        "--model-source", MODEL_SOURCE,
        "--input-file", str(INPUT_FILE),
        "--output-file", str(output_file),
        "--entity-types", "PER,LOC",
        "--limit", str(limit),
        "--batch-size", "16",
        "--max-length", "512",
        "--stride", "128",
        "--device", "cuda",
    ]
    if MODEL_SOURCE == "huggingface":
        command += ["--model-name", MODEL_NAME]
    else:
        command += ["--model-path", str(MODEL_PATH)]
    return command

assert INPUT_FILE.exists(), "Change INPUT_FILE in the configuration cell to your attached Kaggle dataset path."
sample_command = make_command(SAMPLE_OUTPUT, SAMPLE_LIMIT)
print(" ".join(sample_command))
subprocess.run(sample_command, check=True)

In [ ]:
# Inspect the first predictions without loading the complete output into memory.
with SAMPLE_OUTPUT.open(encoding="utf-8") as handle:
    sample_predictions = [json.loads(line) for _, line in zip(range(3), handle)]

for document in sample_predictions:
    print("\nDOC:", document["doc_id"], "characters:", document["char_len"])
    if document.get("error"):
        print("ERROR:", document["error"])
    for entity in document["entities"]:
        print(f"  {entity['label']:>3} | {entity['score']:.3f} | {entity['text']} \[{entity['start']}:{entity['end']}\]")

In [ ]:
# Summarize the sample. This is prediction volume, not precision/recall/F1.
entity_counts = Counter()
documents = 0
documents_with_entities = 0
with SAMPLE_OUTPUT.open(encoding="utf-8") as handle:
    for line in handle:
        document = json.loads(line)
        documents += 1
        entities = document.get("entities", [])
        if entities:
            documents_with_entities += 1
        entity_counts.update(entity["label"] for entity in entities)

print({
    "documents": documents,
    "documents_with_entities": documents_with_entities,
    "entity_counts": dict(entity_counts),
})

In [ ]:
# Run all 10,000 documents only after reviewing the sample.
if RUN_FULL_DATASET:
    full_command = make_command(FULL_OUTPUT, 0)
    print(" ".join(full_command))
    subprocess.run(full_command, check=True)
else:
    print("Full run skipped. Set RUN_FULL_DATASET = True and rerun this cell when ready.")

In [ ]:
# Kaggle exposes files under /kaggle/working for download from the right panel.
from IPython.display import FileLink, display

result_to_download = FULL_OUTPUT if FULL_OUTPUT.exists() else SAMPLE_OUTPUT
display(FileLink(str(result_to_download)))
print("Saved:", result_to_download)